# EOG historical financial record

**Scope:** standardize five years of EOG financial and operating history (FY2021–FY2025). This notebook stops before forecasts, WACC, terminal value, or a DCF.

The main financial table uses EOG's audited annual reports for consistent statement definitions and USD-million scale. The saved WRDS/Compustat extract is retained for source comparison; it is not silently mixed into filing-based calculations. Working capital is not yet standardized to a DCF-ready definition.


## 1. Load the project and latest WRDS report

In [ ]:
from pathlib import Path
from decimal import Decimal
import json
import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "app" / "wrds_client.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find app/wrds_client.py. Start JupyterLab from the DCF-Model project folder.")

report_files = sorted((PROJECT_ROOT / "data" / "raw" / "wrds").glob("*/verification.json"))
if not report_files:
    raise FileNotFoundError("No WRDS report found. See notebooks/README.md to retrieve one first.")
report_path = report_files[-1]
report = json.loads(report_path.read_text(encoding="utf-8"))
compustat = report["compustat"]
if compustat["status"] != "available":
    raise RuntimeError(f"Latest WRDS report status is {compustat['status']}; inspect its attempts before proceeding.")

records = sorted(compustat["records"], key=lambda row: int(row["fyear"]))
years = [int(row["fyear"]) for row in records]
by_year = {int(row["fyear"]): row for row in records}
print(f"Source: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Provider/table: WRDS / {compustat['table']}")
print(f"Retrieved UTC: {report['retrieved_at_utc']}")
print(f"Fiscal years: {years}")

## 2. Standardize the audited historical statements

The annual reports present consolidated statement amounts in USD millions. Filing values below are the primary historical series. EBITDA is calculated as reported operating income plus DD&A; it is not EOG's adjusted EBITDA measure. Capital expenditures are cash additions to oil and gas properties plus other PP&E. They exclude acquisitions and asset-sale proceeds.


In [ ]:
filing = {
    "revenue": [18642, 25702, 24186, 23698, 22632],
    "operating_expenses": [12540, 15736, 14583, 15616, 16247],
    "ebit": [6102, 9966, 9603, 8082, 6385],
    "dd_a": [3651, 3542, 3492, 4108, 4461],
    "interest_expense_net": [178, 179, 148, 138, 235],
    "pretax_income": [5933, 9901, 9689, 8218, 6362],
    "income_tax": [1269, 2142, 2095, 1815, 1382],
    "net_income": [4664, 7759, 7594, 6403, 4980],
    "operating_cash_flow": [8791, 11093, 11340, 12143, 10044],
    "capex": [3850, 5000, 6185, 6372, 6594],
    "cash": [5209, 5972, 5278, 7092, 3396],
    "diluted_shares": [584, 587, 584, 569, 546],
}

def wrds_amount(year, field):
    value = by_year[year].get(field)
    return None if value in (None, "") else Decimal(str(value))

def filed_value(key, year):
    return filing[key][years.index(year)]

def filed_ebitda(year):
    i = years.index(year)
    return filing["ebit"][i] + filing["dd_a"][i]

def compustat_debt(year):
    current = wrds_amount(year, "dlc")
    long_term = wrds_amount(year, "dltt")
    return None if current is None or long_term is None else current + long_term

history_rows = [
    ("Revenue", "EOG reported", "Consolidated Statements of Income: Operating Revenues and Other", "USD millions", lambda y: filed_value("revenue", y)),
    ("Operating expenses", "EOG reported", "Consolidated Statements of Income: total operating expenses", "USD millions", lambda y: filed_value("operating_expenses", y)),
    ("EBITDA", "Calculated", "EOG reported operating income + EOG DD&A", "USD millions", filed_ebitda),
    ("D&A / DD&A", "EOG reported", "Consolidated Statements of Cash Flows: DD&A", "USD millions", lambda y: filed_value("dd_a", y)),
    ("EBIT / operating income", "EOG reported", "Consolidated Statements of Income: Operating Income", "USD millions", lambda y: filed_value("ebit", y)),
    ("Interest expense, net", "EOG reported", "Consolidated Statements of Income", "USD millions", lambda y: filed_value("interest_expense_net", y)),
    ("Pretax income", "EOG reported", "Consolidated Statements of Income: Income Before Income Taxes", "USD millions", lambda y: filed_value("pretax_income", y)),
    ("Income tax provision", "EOG reported", "Consolidated Statements of Income", "USD millions", lambda y: filed_value("income_tax", y)),
    ("Net income", "EOG reported", "Consolidated Statements of Income", "USD millions", lambda y: filed_value("net_income", y)),
    ("Operating cash flow", "EOG reported", "Consolidated Statements of Cash Flows", "USD millions", lambda y: filed_value("operating_cash_flow", y)),
    ("Capital expenditures", "Calculated", "Cash additions to oil and gas properties + other PP&E", "USD millions", lambda y: filed_value("capex", y)),
    ("Change in operating NWC", "Unavailable", "Needs a defined operating NWC balance series; cash flow rows include other items", "USD millions", lambda y: None),
    ("Cash and cash equivalents", "EOG reported", "Consolidated Balance Sheets", "USD millions", lambda y: filed_value("cash", y)),
    ("Total debt", "WRDS reported components", "Compustat dlc + dltt; reconcile definition to filed debt and leases", "USD millions", compustat_debt),
    ("Diluted weighted-average shares", "EOG reported", "Consolidated Statements of Income", "Millions of shares", lambda y: filed_value("diluted_shares", y)),
]
history_table = pd.DataFrame([
    {"Metric": metric, "Value type": kind, "Source / formula": source, "Units": units,
     **{str(year): value(year) for year in years}}
    for metric, kind, source, units, value in history_rows
])
display(history_table)


### Notes on definitions and units

- Reported statement amounts and cash additions are in USD millions; share counts are in millions. Values come from EOG's audited annual reports: [2023 Form 10-K](https://www.sec.gov/Archives/edgar/data/821189/000082118924000011/eog-20231231.htm) supplies FY2021–FY2023 comparative statements, and [2025 Form 10-K](https://www.sec.gov/Archives/edgar/data/821189/000082118926000054/eog-20251231.htm) supplies FY2023–FY2025. FY2023 overlaps as a consistency check.
- EBITDA here means operating income plus DD&A. It is a simple calculated measure, not EOG's adjusted EBITDA.
- Capital expenditures are additions to oil and gas properties plus additions to other property, plant and equipment. They exclude the 2025 Encino acquisition and asset-sale proceeds.
- Debt remains a Compustat-derived series (`dlc + dltt`) and is clearly labeled because its classification does not match the face of EOG's balance sheet in every year. Reconcile current maturities, finance leases, and Compustat definitions before valuation use.
- The company cash-flow statement's working-capital and other-assets/liabilities section is not a clean change in operating NWC series. It combines accounts receivable, inventory, accounts payable, accrued taxes, other assets/liabilities, and a separate investing-related working-capital item. Therefore, DCF-ready change in NWC remains blank rather than treating that broad cash-flow effect as operating NWC.


## 3. Calculate historical metrics

These historical ratios use the audited filing series above. UFCF stays blank until a DCF-ready working-capital definition is agreed and calculated.


In [ ]:
def ratio(numerator, denominator):
    if denominator in (None, 0):
        return None
    return numerator / denominator

metric_specs = [
    ("Revenue growth", "Current-year revenue / prior-year revenue - 1", "%", lambda i: None if i == 0 else ratio(filing["revenue"][i], filing["revenue"][i-1]) - 1),
    ("EBITDA margin", "(Operating income + DD&A) / revenue", "%", lambda i: ratio(filed_ebitda(years[i]), filing["revenue"][i])),
    ("EBIT margin", "Operating income / revenue", "%", lambda i: ratio(filing["ebit"][i], filing["revenue"][i])),
    ("Capex / revenue", "Capital expenditures / revenue", "%", lambda i: ratio(filing["capex"][i], filing["revenue"][i])),
    ("Effective tax rate", "Income tax provision / pretax income", "%", lambda i: ratio(filing["income_tax"][i], filing["pretax_income"][i])),
    ("Historical UFCF", "EBIT × (1 - tax rate) + DD&A - capex - change in operating NWC", "USD millions; unavailable", lambda i: None),
]
metric_table = pd.DataFrame([
    {"Metric": name, "Formula": formula, "Units": unit,
     **{str(year): (None if calc(i) is None else calc(i) * 100 if unit == "%" else calc(i)) for i, year in enumerate(years)}}
    for name, formula, unit, calc in metric_specs
])
display(metric_table)


Operating cash flow is reported separately and is not substituted for UFCF. The missing input is a consistent, defined change in operating NWC series; the statement's broad working-capital cash-flow reconciliation is not equivalent.


## 4. Add five years of E&P operating history

Production and pricing values are EOG-reported annual statistics. Costs are reported dollars per Boe; the transportation/processing series is made comparable by summing EOG's separate transportation and gathering/processing rows for 2021–2023, then using EOG's combined GP&T row for 2024–2025. Reserve balances are year-end net proved reserves. Compustat capex is in USD millions. A dash reported by EOG for an operating volume is treated as zero only where the table explicitly presents no production for that category; otherwise missing values remain blank.

Sources: [EOG 2022 Form 10-K](https://www.sec.gov/Archives/edgar/data/821189/000082118923000015/eog-20221231.htm), [EOG 2023 Form 10-K](https://www.sec.gov/Archives/edgar/data/821189/000082118924000011/eog-20231231.htm), [EOG 2024 Form 10-K](https://www.sec.gov/Archives/edgar/data/821189/000082118925000011/eog-20241231.htm), [EOG 2025 Form 10-K](https://www.sec.gov/Archives/edgar/data/821189/000082118926000054/eog-20251231.htm), and EIA's [annual Cushing WTI spot-price history](https://www.eia.gov/dnav/pet/hist/rwtca.htm).

In [ ]:
operating_rows = [
    ("Oil production", "MBbl/day", "EOG reported total crude oil and condensate production", [445.0, 461.3, 475.8, 491.4, 521.9]),
    ("Natural gas production", "MMcf/day", "EOG reported total natural gas production", [1436, 1495, 1711, 1948, 2533]),
    ("NGL production", "MBbl/day", "EOG reported total NGL production", [144.5, 197.7, 223.8, 245.9, 288.2]),
    ("Total production", "MBoE/day", "EOG reported total crude-oil-equivalent production", [828.9, 908.2, 984.8, 1062.1, 1232.2]),
    ("Total annual production", "MMBoe/year", "EOG reported annual total; not recalculated", [302.5, 331.5, 359.4, 388.7, 449.8]),
    ("Realized oil price", "USD/Bbl", "EOG composite crude oil and condensate price; excludes financial derivatives", [68.50, 97.21, 79.17, 77.40, 65.63]),
    ("Realized natural gas price", "USD/Mcf", "EOG composite natural gas price; excludes financial derivatives", [4.66, 6.93, 2.79, 2.17, 3.02]),
    ("Realized NGL price", "USD/Bbl", "EOG composite NGL price; excludes financial derivatives", [34.35, 36.70, 23.07, 23.40, 22.58]),
    ("Lease and well expense", "USD/Boe", "EOG reported cost per Boe", [3.75, 4.02, 4.05, 4.04, 3.72]),
    ("Gathering, processing and transportation", "USD/Boe", "2021–23: transportation + gathering/processing; 2024–25: reported combined GP&T", [4.70, 4.78, 4.50, 4.43, 4.74]),
    ("General and administrative", "USD/Boe", "EOG reported cost per Boe", [1.69, 1.72, 1.78, 1.72, 1.82]),
    ("Taxes other than income", "USD millions", "Includes production/severance, property, payroll, franchise and other taxes", [1047, 1585, 1284, 1249, 1234]),
    ("Capital expenditures", "USD millions", "EOG cash additions to oil and gas properties + other PP&E", filing["capex"]),
    ("Proved crude oil reserves", "MMBbl at year-end", "EOG net proved reserves", [1548, 1661, 1756, 1870, 1905]),
    ("Proved NGL reserves", "MMBbl at year-end", "EOG net proved reserves", [829, 1145, 1254, 1358, 1510]),
    ("Proved natural gas reserves", "Bcf at year-end", "EOG net proved reserves", [8222, 8591, 8930, 9122, 12592]),
    ("Total proved reserves", "MMBoe at year-end", "EOG net proved oil-equivalent reserves", [3747, 4238, 4498, 4748, 5514]),
    ("WTI benchmark", "USD/Bbl", "EIA annual average Cushing WTI spot price", [68.13, 94.90, 77.58, 76.63, 65.39]),
]
operating_table = pd.DataFrame([
    {"Metric": metric, "Units": units, "Source / definition": note,
     **{str(year): value for year, value in zip(years, values)}}
    for metric, units, note, values in operating_rows
])
display(operating_table)


## 5. Calculate business-driver metrics

The production, price, cost, reserve, and benchmark data above feed the descriptive metrics below. These are historical diagnostics, not forecast assumptions.

### Source and comparability notes

- EOG operating statistics are annual reported figures. Production in MBbl/day or MMcf/day is an average rate; annual MMBoe is shown separately. EOG converts gas to oil equivalent at 6 Mcf per Boe.
- The 2022 composite gas price includes EOG’s disclosed positive $0.09/Mcf revenue adjustment related to an earlier sales contract.
- EOG cost tables exclude exploration, dry-hole costs, impairments, marketing costs and taxes other than income. Transportation/processing headings change across filings; the 2021–2023 combined values are calculated from the two reported rows.
- “Taxes other than income” is broader than production taxes, so it is labeled separately and not treated as a pure production-tax series.
- EOG’s 2025 proved reserves include acquisition activity, including the Encino transaction. The increase should not be read as organic reserve growth.
- WTI is a broad benchmark. EOG’s composite realized crude price reflects its own locations and sales mix, so the differential is indicative, not a basin-specific price spread.


In [ ]:
def metric_values(values):
    return {str(year): values[i] for i, year in enumerate(years)}

oil = [445.0, 461.3, 475.8, 491.4, 521.9]
gas = [1436, 1495, 1711, 1948, 2533]
gas_boe = [value / 6 for value in gas]
ngl = [144.5, 197.7, 223.8, 245.9, 288.2]
total_mboed = [828.9, 908.2, 984.8, 1062.1, 1232.2]
annual_mmboe = [302.5, 331.5, 359.4, 388.7, 449.8]
wti = [68.13, 94.90, 77.58, 76.63, 65.39]
realized_oil = [68.50, 97.21, 79.17, 77.40, 65.63]

business_metrics = [
    {"Metric": "Oil mix of total production", "Formula / source": "Oil MBbl/day / reported total MBoE/day", "Units": "%", **metric_values([100*x/t for x,t in zip(oil,total_mboed)])},
    {"Metric": "Natural gas mix of total production", "Formula / source": "(MMcf/day / 6) / reported total MBoE/day", "Units": "%", **metric_values([100*x/t for x,t in zip(gas_boe,total_mboed)])},
    {"Metric": "NGL mix of total production", "Formula / source": "NGL MBbl/day / reported total MBoE/day", "Units": "%", **metric_values([100*x/t for x,t in zip(ngl,total_mboed)])},
    {"Metric": "Production growth", "Formula / source": "Year-over-year change in reported annual MMBoe", "Units": "%", **metric_values([None] + [100*(annual_mmboe[i]/annual_mmboe[i-1]-1) for i in range(1,len(years))])},
    {"Metric": "LOE / Boe", "Formula / source": "EOG reported lease and well cost", "Units": "USD/Boe", **metric_values([3.75,4.02,4.05,4.04,3.72])},
    {"Metric": "Gathering, processing and transportation / Boe", "Formula / source": "Normalized reported cost series; see source note", "Units": "USD/Boe", **metric_values([4.70,4.78,4.50,4.43,4.74])},
    {"Metric": "G&A / Boe", "Formula / source": "EOG reported G&A cost", "Units": "USD/Boe", **metric_values([1.69,1.72,1.78,1.72,1.82])},
    {"Metric": "Capex / Boe", "Formula / source": "EOG-reported cash additions to PP&E / annual production", "Units": "USD/Boe", **metric_values([filing["capex"][i]/annual_mmboe[i] for i in range(len(years))])},
    {"Metric": "Calculated EBITDA / Boe", "Formula / source": "(EOG reported operating income + EOG DD&A) / annual production", "Units": "USD/Boe", **metric_values([filed_ebitda(year)/annual_mmboe[i] for i,year in enumerate(years)])},
    {"Metric": "WTI less realized oil price", "Formula / source": "EIA annual average Cushing WTI - EOG composite realized price", "Units": "USD/Bbl", **metric_values([benchmark-realized for benchmark,realized in zip(wti,realized_oil)])},
]
business_metrics_table = pd.DataFrame(business_metrics)
display(business_metrics_table)


## 6. Compare filing values with WRDS / Compustat

This comparison explains why the notebook uses the audited filing series as its primary historical record. Compustat field definitions and displayed values are retained as a separate vendor series; differences should be reconciled before combining them with filing data.



In [ ]:
comparison_fields = [
    ("Revenue", "revt", filing["revenue"]),
    ("Operating income", "oiadp", filing["ebit"]),
    ("D&A", "dp", filing["dd_a"]),
    ("Capital expenditures", "capx", filing["capex"]),
    ("Operating cash flow", "oancf", filing["operating_cash_flow"]),
]
comparison_rows = []
for label, field, filed_values in comparison_fields:
    for i, year in enumerate(years):
        wrds_value = wrds_amount(year, field)
        filing_value = filed_values[i]
        comparison_rows.append({
            "Metric": label, "Fiscal year": year,
            "WRDS / Compustat": wrds_value,
            "EOG filing": filing_value,
            "WRDS less filing": None if wrds_value is None else wrds_value - Decimal(str(filing_value)),
        })
wrds_comparison = pd.DataFrame(comparison_rows)
display(wrds_comparison)


## What the history says so far

Use the tables above to describe past results, not to select forecast assumptions yet. Production grew through this period, while commodity prices, unit costs and the proved-reserve base moved for different reasons. The reserve increase in 2025 includes acquisition effects. Before forecasting, finish the missing statement items and review units/source definitions, then assess the operating trends with that context.

## What is complete and what comes next

The historical statement and operating-driver data are organized, and the primary financial series now uses consistent audited filing definitions. The next step is to review the WRDS-to-filing reconciliation, resolve debt classification and working-capital definitions, and then write a short historical trend assessment. Forecast assumptions and the DCF come after that review.


In [ ]:
from IPython.display import display, Markdown

if list(years) != [2021, 2022, 2023, 2024, 2025]:
    raise ValueError("These source values are prepared for FY2021–FY2025.")

def vendor_value(year, field):
    value = by_year[year].get(field)
    return float(value) if value not in (None, "") else float("nan")

field_mapping = [
    ("Revenue", "revt", "revenue"),
    ("Operating income", "oiadp", "ebit"),
    ("DD&A", "dp", "dd_a"),
    ("Capital expenditures", "capx", "capex"),
    ("Operating cash flow", "oancf", "operating_cash_flow"),
]

reconciliation_rows = []

for label, vendor_field, filing_field in field_mapping:
    for i, year in enumerate(years):
        vendor = vendor_value(year, vendor_field)
        reported = float(filing[filing_field][i])
        difference = vendor - reported

        status = (
            "Missing vendor value" if pd.isna(vendor)
            else "Matches" if abs(difference) < 0.01
            else "Definition bridge needed"
        )

        reconciliation_rows.append({
            "Metric": label,
            "Year": year,
            "Compustat": vendor,
            "EOG filing": reported,
            "Difference": difference,
            "Review status": status,
        })

financial_reconciliation = pd.DataFrame(reconciliation_rows)

print("Amounts and differences are USD millions; difference = Compustat − filing.")
display(financial_reconciliation.round(2))